# Laboratorio 10 · Explicabilidad en datos tabulares con SHAP, LIME y contrafactuales

Este notebook se sitúa en el equipo de riesgo de modelo de una entidad financiera que debe justificar decisiones de crédito. Sobre un clasificador de gradient boosting entrenado con German Credit se calculan valores SHAP y se verifican con la fórmula de Shapley (Task 2), se implementa LIME desde cero y se construyen explicaciones contrafactuales (Task 3).

Las funciones reutilizables se encuentran en `src/tabular/`. El notebook está pensado para ejecutarse en Google Colab.

## Configuración

La siguiente celda clona el repositorio cuando el notebook se ejecuta en Colab, agrega su carpeta raíz al `sys.path` e instala la librería `shap`.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Tunchxz/CC3092-Laboratorio-10.git"
ROOT = "/content/CC3092-Laboratorio-10"

if "google.colab" in sys.modules:
    if not os.path.isdir(ROOT):
        !git clone -q {REPO_URL} {ROOT}
    %pip install -q shap
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score

from src.tabular.data import SEED, decode, load_credit, log_odds_fn, proba_fn, split, train_gbm
from src.tabular.shap_utils import tree_shap
from src.tabular.shapley import coalition_values, shapley_from_values
from src.tabular.lime_tabular import lime_explain, mean_pairwise_jaccard
from src.tabular.counterfactuals import distance, generate_candidates, mad_scale, select_diverse

warnings.filterwarnings("ignore", category=FutureWarning)
rng = np.random.default_rng(SEED)
pd.set_option("display.float_format", lambda v: f"{v:.6g}")
print(f"Semilla aleatoria: {SEED} | shap {shap.__version__}")


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## Task 2 · SHAP

### Task 2.1 · Datos y modelo

Se descarga German Credit con `fetch_openml`. La clase `good` se codifica como 1 y `bad` como 0, y las 13 variables categóricas se codifican con `OrdinalEncoder`.

> **Limitación.** Los códigos ordinales son arbitrarios (por ejemplo, el orden alfabético de las categorías), pero el modelo los trata como números. Los árboles pueden cortar entre códigos que no tienen un orden real, y las explicaciones que hablan de "valores altos" o "valores bajos" de una variable categórica no tienen un significado sustantivo.

El conjunto se divide en 80% de entrenamiento y 20% de prueba de forma estratificada, y se entrena un `GradientBoostingClassifier` con los siguientes valores.

| Parámetro | Valor |
| :-------- | :---: |
| Semilla aleatoria | 42 |
| Número de árboles | 200 |
| Profundidad máxima de cada árbol | 3 |
| Tamaño del fondo para SHAP | 100 instancias |

In [ ]:
X, y, encoder, categorical, numeric = load_credit()
cols = list(X.columns)
print(f"{X.shape[0]} solicitudes, {X.shape[1]} variables ({len(numeric)} numéricas y {len(categorical)} categóricas)")
print(f"Numéricas: {numeric}")
X.head()

In [ ]:
X_train, X_test, y_train, y_test = split(X, y)
model = train_gbm(X_train, y_train)
f_logit = log_odds_fn(model, cols)
f_prob = proba_fn(model, cols)

p_test = model.predict_proba(X_test)[:, 1]
print(f"Entrenamiento: {len(X_train)} | prueba: {len(X_test)}")
print(f"Exactitud en prueba: {accuracy_score(y_test, (p_test >= 0.5).astype(int)):.4f}")
print(f"AUC en prueba:       {roc_auc_score(y_test, p_test):.4f}")

### Task 2.2 · Valores SHAP con TreeSHAP intervencional

Se calculan los valores SHAP del conjunto de prueba con `shap.TreeExplainer` en su versión intervencional, sobre la salida en log-odds (`model_output="raw"`) y con un fondo de 100 instancias de entrenamiento elegidas al azar.

In [ ]:
bg_idx = rng.choice(len(X_train), size=100, replace=False)
background = X_train.iloc[bg_idx]

explanation, phi0 = tree_shap(model, X_test, background)
phi = explanation.values
f_test = model.decision_function(X_test)

print("1. Eficiencia")
print(f"   max |phi_0 + sum_j phi_j - f(x)| = {np.abs(phi0 + phi.sum(axis=1) - f_test).max():.3e}")
print("2. Valor base")
print(f"   phi_0                              = {phi0:.6f}")
print(f"   promedio de f(x) sobre el fondo    = {f_logit(background).mean():.6f}")

**3. Importancia global.** Se compara $I_{j} = \frac{1}{N}\sum_{n}\left|\phi_{j}^{(n)}\right|$ con la importancia por permutación de scikit-learn sobre el conjunto de prueba (métrica AUC, 10 repeticiones).

In [ ]:
perm = permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=SEED)
importance = pd.DataFrame({
    "I_j (SHAP)": np.abs(phi).mean(axis=0),
    "permutación (media)": perm.importances_mean,
    "permutación (desv. est.)": perm.importances_std,
}, index=cols)
importance["ranking SHAP"] = importance["I_j (SHAP)"].rank(ascending=False).astype(int)
importance["ranking permutación"] = importance["permutación (media)"].rank(ascending=False).astype(int)

top5 = pd.DataFrame({
    "SHAP": importance["I_j (SHAP)"].nlargest(5).index,
    "permutación": importance["permutación (media)"].nlargest(5).index,
}, index=pd.RangeIndex(1, 6, name="posición"))
display(top5)
rho_rank = spearmanr(importance["I_j (SHAP)"], importance["permutación (media)"]).statistic
print(f"Correlación de Spearman entre los dos rankings completos: {rho_rank:.4f}")
importance.sort_values("ranking SHAP")

**4. Explicaciones locales.** Se elige el primer cliente de prueba con probabilidad de `good` mayor que 0.8 y el primero con probabilidad menor que 0.2. En las cascadas, las variables categóricas se muestran decodificadas.

In [ ]:
i_good = int(np.flatnonzero(p_test > 0.8)[0])
i_bad = int(np.flatnonzero(p_test < 0.2)[0])

X_test_display = X_test.copy().astype(object)
for col in categorical:
    X_test_display[col] = [decode(v, col, encoder, categorical) for v in X_test[col]]
explanation_display = shap.Explanation(values=phi, base_values=np.full(len(phi), phi0), data=X_test.values,
                                       display_data=X_test_display.values, feature_names=cols)

for label, i in (("aprobado", i_good), ("rechazado", i_bad)):
    print(f"Cliente {label}: fila {X_test.index[i]} del conjunto original, posición {i} en prueba, "
          f"p(good) = {p_test[i]:.4f}, f(x) = {f_test[i]:.4f}, clase real = {'good' if y_test.iloc[i] else 'bad'}")
    top3 = np.argsort(-np.abs(phi[i]))[:3]
    display(pd.DataFrame({
        "variable": [cols[j] for j in top3],
        "valor": [X_test_display.iloc[i, j] for j in top3],
        "phi_j": phi[i, top3],
    }))
    shap.plots.waterfall(explanation_display[i], max_display=12, show=False)
    plt.title(f"Cliente {label} (p(good) = {p_test[i]:.3f})")
    plt.show()

In [ ]:
shap.plots.beeswarm(explanation, max_display=20, show=False)
plt.title("Valores SHAP del conjunto de prueba")
plt.show()

### Task 2.3 · Variables redundantes

Se toma la variable numérica de mayor $I_{j}$ y se crea una copia con ruido, $x_{dup} = x + 0.05\,\sigma_{x}\,\epsilon$, con $\epsilon \sim \mathcal{N}(0,1)$. Se reentrena el modelo con 21 variables y la misma semilla, y se calculan los valores SHAP con la versión intervencional (mismo fondo) y con la dependiente de la trayectoria del árbol (`tree_path_dependent`), que no usa fondo.

In [ ]:
top_num = importance.loc[numeric, "I_j (SHAP)"].idxmax()
dup = f"{top_num}_dup"
noise = np.random.default_rng(SEED).standard_normal(len(X))
X_dup = X.copy()
X_dup[dup] = X[top_num] + 0.05 * X[top_num].std() * noise
print(f"Variable duplicada: {top_num} | correlación con la copia: {np.corrcoef(X_dup[top_num], X_dup[dup])[0, 1]:.5f}")

Xd_train, Xd_test, yd_train, _ = split(X_dup, y)
assert Xd_train.index.equals(X_train.index)
model_dup = train_gbm(Xd_train, yd_train)
print(f"AUC en prueba del modelo con 21 variables: {roc_auc_score(y_test, model_dup.predict_proba(Xd_test)[:, 1]):.4f}")

expl_int, _ = tree_shap(model_dup, Xd_test, Xd_train.loc[background.index])
expl_tpd, _ = tree_shap(model_dup, Xd_test)
j_orig, j_dup = Xd_test.columns.get_loc(top_num), Xd_test.columns.get_loc(dup)

rows = []
for name, values in (("intervencional", expl_int.values), ("tree_path_dependent", expl_tpd.values)):
    rows.append({
        "configuración": name,
        "mean |phi_orig|": np.abs(values[:, j_orig]).mean(),
        "mean |phi_copia|": np.abs(values[:, j_dup]).mean(),
        "mean |phi_orig + phi_copia|": np.abs(values[:, j_orig] + values[:, j_dup]).mean(),
    })
dup_table = pd.DataFrame(rows).set_index("configuración")
dup_table["I_j sin copia"] = importance.loc[top_num, "I_j (SHAP)"]
dup_table

### Task 2.4 · KernelSHAP frente a TreeSHAP

Se calculan los valores SHAP de las 10 primeras instancias de prueba con `shap.KernelExplainer` sobre la salida en log-odds, con 2 000 muestras por instancia y sin selección de variables (`l1_reg=False`), para fondos de 10, 50 y 100 instancias (los primeros elementos del fondo del Task 2.2). Cada resultado se compara con TreeSHAP intervencional calculado con el mismo fondo.

In [ ]:
X10 = X_test.iloc[:10]
kernel_rows = []
for size in (10, 50, 100):
    bg = background.iloc[:size]
    start = time.perf_counter()
    kernel = shap.KernelExplainer(f_logit, bg.values)
    phi_kernel = np.asarray(kernel.shap_values(X10.values, nsamples=2000, l1_reg=False, silent=True))
    t_kernel = time.perf_counter() - start

    start = time.perf_counter()
    expl_tree, _ = tree_shap(model, X10, bg)
    t_tree = time.perf_counter() - start

    kernel_rows.append({
        "fondo": size,
        "MAE KernelSHAP vs TreeSHAP": np.abs(phi_kernel.reshape(expl_tree.values.shape) - expl_tree.values).mean(),
        "tiempo KernelSHAP (s)": t_kernel,
        "tiempo TreeSHAP (s)": t_tree,
        "tiempo KernelSHAP por instancia (s)": t_kernel / len(X10),
    })
kernel_table = pd.DataFrame(kernel_rows).set_index("fondo")
kernel_table

### Task 2.5 · Shapley exacto con la fórmula

Se entrena un segundo modelo con las mismas condiciones y solo seis variables numéricas. Para 5 instancias de prueba y un fondo de 50 instancias se define

$$v(S) = \frac{1}{|B|}\sum_{b \in B} f\big(x_{S}, b_{\bar{S}}\big),$$

se enumeran las $2^{6} = 64$ coaliciones y se aplica

$$\phi_{i} = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(n - |S| - 1)!}{n!}\,\big[v(S \cup \{i\}) - v(S)\big].$$

Las funciones `coalition_values` y `shapley_from_values` de `src/tabular/shapley.py` implementan ambos pasos sin usar `shap`.

In [ ]:
NUM6 = ["duration", "credit_amount", "installment_commitment", "residence_since", "age", "existing_credits"]
model6 = train_gbm(X_train[NUM6], y_train)
f6 = log_odds_fn(model6, NUM6)
bg50 = X_train[NUM6].iloc[bg_idx[:50]]
X5 = X_test[NUM6].iloc[:5]

phi_exact, efficiency = [], []
for x in X5.values:
    v = coalition_values(f6, x, bg50.values)
    phi_i = shapley_from_values(v, len(NUM6))
    phi_exact.append(phi_i)
    efficiency.append(abs(phi_i.sum() - (v[-1] - v[0])))
phi_exact = np.array(phi_exact)

expl6, _ = tree_shap(model6, X5, bg50)
print(f"Error máximo |phi exacto - phi TreeSHAP|:   {np.abs(phi_exact - expl6.values).max():.3e}")
print(f"Error máximo de eficiencia |sum phi - (v(N) - v(vacío))|: {max(efficiency):.3e}")
pd.concat({"exacto": pd.DataFrame(phi_exact, columns=NUM6),
           "TreeSHAP": pd.DataFrame(expl6.values, columns=NUM6)}, axis=1).rename_axis("instancia")

### Task 2.6 · Análisis con los resultados

#### 2.6.1 · De log-odds a probabilidad en el cliente rechazado

In [ ]:
p_x = sigmoid(f_test[i_bad])
p_base = sigmoid(phi0)
effects = sigmoid(phi0 + phi[i_bad]) - p_base
print(f"p(x) = sigma(f(x))            = {p_x:.6f}")
print(f"sigma(phi_0)                  = {p_base:.6f}")
print(f"suma de efectos individuales  = {effects.sum():.6f}")
print(f"p(x) - sigma(phi_0)           = {p_x - p_base:.6f}")
pd.DataFrame({"phi_j (log-odds)": phi[i_bad], "efecto en probabilidad": effects}, index=cols) \
    .sort_values("phi_j (log-odds)", key=np.abs, ascending=False).head(8)

#### 2.6.2 · Variable con mayor diferencia de ranking

In [ ]:
importance["diferencia de ranking"] = (importance["ranking SHAP"] - importance["ranking permutación"]).abs()
var_gap = importance["diferencia de ranking"].idxmax()
print(f"Variable con mayor diferencia de posición: {var_gap}")
display(importance.loc[[var_gap]])

corr = X_train.corr(method="spearman")[var_gap].drop(var_gap)
print(f"Correlaciones de Spearman de '{var_gap}' con las demás variables (entrenamiento), top 5 en valor absoluto:")
corr.reindex(corr.abs().sort_values(ascending=False).index).head(5).to_frame("Spearman")

#### 2.6.4 · Costo computacional

In [ ]:
kernel_table["evaluaciones (2000 x fondo)"] = 2000 * kernel_table.index
kernel_table["tiempo por evaluación (µs)"] = 1e6 * kernel_table["tiempo KernelSHAP por instancia (s)"] / kernel_table["evaluaciones (2000 x fondo)"]
display(kernel_table[["evaluaciones (2000 x fondo)", "tiempo KernelSHAP por instancia (s)",
                      "tiempo por evaluación (µs)", "MAE KernelSHAP vs TreeSHAP"]])

X100 = X_test.iloc[:100]
repeats = 50
start = time.perf_counter()
for _ in range(repeats):
    f_logit(X100)
t_100 = (time.perf_counter() - start) / repeats

t_exact_20 = 2 ** 20 * t_100
t_kernel_100 = kernel_table.loc[100, "tiempo KernelSHAP por instancia (s)"]
years_60 = 2 ** 60 * 1e-6 / (365.25 * 24 * 3600)
print(f"Tiempo de una predicción sobre 100 instancias:        {t_100 * 1e3:.3f} ms")
print(f"Shapley exacto, 20 variables (2^20 x t_100):          {t_exact_20:.1f} s = {t_exact_20 / 3600:.2f} h")
print(f"KernelSHAP por instancia con fondo de 100:            {t_kernel_100:.2f} s")
print(f"Razón exacto / KernelSHAP:                            {t_exact_20 / t_kernel_100:.0f}")
print(f"Shapley exacto, 60 variables, 1 µs por evaluación:    {years_60:,.0f} años")

## Task 3 · LIME y contrafactuales

### Task 3.1 · LIME desde cero

La representación interpretable usa las 20 variables originales. Cada variable está presente (conserva el valor de la instancia) o ausente (toma un valor al azar de su columna en entrenamiento). La función `lime_explain` de `src/tabular/lime_tabular.py` sigue estos pasos.

1. Genera $N$ vectores binarios $z'$ con componentes iguales a 1 con probabilidad 0.5 y construye las entradas $z$.
2. Explica la probabilidad de `good` y pondera cada muestra con $\pi_{x}(z) = \exp\left(-\frac{D^{2}}{\sigma^{2}}\right)$, donde $D^{2}$ es el número de variables ausentes y $\sigma = \sigma_{rel}\sqrt{d'}$ con $d' = 20$.
3. Ajusta $w = \left({Z'}^{\intercal}\Pi Z' + \lambda I\right)^{-1}{Z'}^{\intercal}\Pi y$ con una columna de unos para el intercepto. **El intercepto no se penaliza** (su entrada en $\lambda I$ es cero).
4. Conserva las $K$ variables de mayor $|w|$, vuelve a ajustar solo con ellas y calcula la fidelidad ponderada $R_{w}^{2}$.

| Parámetro | Valor |
| :-------- | :---: |
| Número de muestras $N$ | 5 000 |
| Variables que se conservan $K$ | 5 |
| Regularización $\lambda$ | 1.0 |
| Ancho relativo del kernel $\sigma_{rel}$ | 0.75 |

Se explican tres clientes de prueba: el aprobado del Task 2.2 (probabilidad mayor que 0.8), el rechazado del Task 2.2 y el primero con probabilidad entre 0.45 y 0.55.

In [ ]:
X_train_arr, X_test_arr = X_train.values, X_test.values
mid = np.flatnonzero((p_test > 0.45) & (p_test < 0.55))
i_mid = int(mid[0]) if mid.size else int(np.argmin(np.abs(p_test - 0.5)))
clients = {"aprobado": i_good, "rechazado": i_bad, "frontera": i_mid}

lime_results = {}
for label, i in clients.items():
    lime_results[label] = lime_explain(f_prob, X_test_arr[i], X_train_arr, n_samples=5000, k=5, lam=1.0,
                                       sigma_rel=0.75, rng=np.random.default_rng(SEED))
    r = lime_results[label]
    print(f"Cliente {label} (posición {i}, p(good) = {p_test[i]:.4f}) | intercepto = {r['intercept']:.4f} | R²_w = {r['r2_w']:.4f}")
    display(pd.DataFrame({"variable": [cols[j] for j in r["features"]],
                          "valor": [X_test_display.iloc[i, j] for j in r["features"]],
                          "coeficiente": r["coef"]}))

### Task 3.2 · Estabilidad de LIME en el cliente rechazado

Para cada combinación de $N$ y $\sigma_{rel}$ se ejecuta LIME 20 veces con semillas distintas. Se reporta el índice de Jaccard promedio entre los conjuntos de las $K$ variables principales de todos los pares de corridas y el tamaño efectivo de muestra promedio $N_{ef} = \frac{\left(\sum_{n}\pi_{n}\right)^{2}}{\sum_{n}\pi_{n}^{2}}$.

In [ ]:
N_VALUES = (500, 2000, 5000)
SIGMA_VALUES = (0.5, 0.75, 1.5)
RUN_SEEDS = range(SEED, SEED + 20)
x_bad = X_test_arr[i_bad]


def lime_stability(n_samples, sigma_rel, seeds=RUN_SEEDS):
    runs = [lime_explain(f_prob, x_bad, X_train_arr, n_samples=n_samples, sigma_rel=sigma_rel,
                         rng=np.random.default_rng(s)) for s in seeds]
    return mean_pairwise_jaccard([set(r["features"]) for r in runs]), np.mean([r["n_eff"] for r in runs])


jaccard_table = pd.DataFrame(index=pd.Index(N_VALUES, name="N"), columns=SIGMA_VALUES, dtype=float)
neff_table = jaccard_table.copy()
for n_samples in N_VALUES:
    for sigma_rel in SIGMA_VALUES:
        jaccard_table.loc[n_samples, sigma_rel], neff_table.loc[n_samples, sigma_rel] = lime_stability(n_samples, sigma_rel)
jaccard_table.columns.name = neff_table.columns.name = "sigma_rel"
print("Jaccard promedio")
display(jaccard_table)
print("N_ef promedio")
display(neff_table)

### Task 3.3 · LIME frente a SHAP

**1.** Para cada cliente se calcula la correlación de Spearman entre $|w|$ de LIME (ajuste sobre las 20 variables) y $|\phi|$ de SHAP del Task 2.2, y la coincidencia entre las tres variables principales de cada método.

In [ ]:
rows = []
for label, i in clients.items():
    w_abs = np.abs(lime_results[label]["coef_full"])
    phi_abs = np.abs(phi[i])
    top_lime = [cols[j] for j in np.argsort(-w_abs)[:3]]
    top_shap = [cols[j] for j in np.argsort(-phi_abs)[:3]]
    rows.append({
        "cliente": label,
        "Spearman |w| vs |phi|": spearmanr(w_abs, phi_abs).statistic,
        "top-3 LIME": top_lime,
        "top-3 SHAP": top_shap,
        "coincidencias": len(set(top_lime) & set(top_shap)),
    })
pd.DataFrame(rows).set_index("cliente")

**2.** Para el cliente rechazado se ajusta LIME con $K \in \{3, 5, 10, 20\}$, con las mismas muestras, y se reporta $R_{w}^{2}$.

In [ ]:
r2_by_k = pd.Series({k: lime_explain(f_prob, x_bad, X_train_arr, n_samples=5000, k=k, lam=1.0, sigma_rel=0.75,
                                     rng=np.random.default_rng(SEED))["r2_w"]
                     for k in (3, 5, 10, 20)}, name="R²_w")
r2_by_k.index.name = "K"
r2_by_k.to_frame()

### Task 3.4 · Explicaciones contrafactuales

Se buscan contrafactuales para el cliente rechazado con 20 000 candidatos. Cada candidato cambia un subconjunto aleatorio de entre 1 y 3 variables accionables (`duration`, `credit_amount`, `installment_commitment`, `checking_status`, `savings_status`) con valores observados en entrenamiento. Las variables `age`, `personal_status` y `foreign_worker` son inmutables. Un candidato es válido si su probabilidad de `good` es al menos $\theta = 0.5$, y su distancia al original es

$$d(x, x') = \sum_{j \in \text{num}}\frac{\left|x_{j} - x_{j}'\right|}{\operatorname{MAD}_{j}} + \sum_{j \in \text{cat}}\mathbb{1}\left[x_{j} \neq x_{j}'\right].$$

Se elige el candidato válido más cercano y luego dos más con conjuntos de variables cambiadas distintos. La proximidad y la dispersión se calculan sobre los tres contrafactuales elegidos.

In [ ]:
ACTIONABLE = ["duration", "credit_amount", "installment_commitment", "checking_status", "savings_status"]
IMMUTABLE = ["age", "personal_status", "foreign_worker"]
THETA = 0.5
actionable_idx = [cols.index(c) for c in ACTIONABLE]
numeric_idx = [cols.index(c) for c in numeric]
categorical_idx = [cols.index(c) for c in categorical]
scale = mad_scale(X_train_arr)


def counterfactual_search(max_changes, n_candidates=20000):
    cands = generate_candidates(x_bad, X_train_arr, actionable_idx, n_candidates, max_changes,
                                np.random.default_rng(SEED))
    probs = f_prob(cands)
    return cands, probs, probs >= THETA, distance(x_bad, cands, numeric_idx, categorical_idx, scale)


max_changes = 3
cands, cand_probs, valid, dist = counterfactual_search(max_changes)
if not valid.any():
    max_changes = 5
    print("Ningún candidato válido con 3 cambios; se aumenta el máximo a 5 variables.")
    cands, cand_probs, valid, dist = counterfactual_search(max_changes)
chosen = select_diverse(x_bad, cands, dist, valid, n_select=3)

rows = []
for rank, idx in enumerate(chosen, start=1):
    for j in np.flatnonzero(cands[idx] != x_bad):
        rows.append({
            "contrafactual": rank,
            "variable": cols[j],
            "original": decode(x_bad[j], cols[j], encoder, categorical),
            "nuevo": decode(cands[idx, j], cols[j], encoder, categorical),
            "distancia": dist[idx],
            "p(good)": cand_probs[idx],
        })
print(f"Cliente rechazado: p(good) = {p_test[i_bad]:.4f} | máximo de variables cambiadas = {max_changes}")
print(f"Validez (fracción de candidatos válidos): {valid.mean():.4f} ({valid.sum()} de {len(valid)})")
print(f"Proximidad promedio (distancia media):    {dist[chosen].mean():.4f}")
print(f"Dispersión (variables cambiadas, media):  {np.mean([(cands[i] != x_bad).sum() for i in chosen]):.2f}")
print("MAD de las variables numéricas accionables:",
      {c: round(scale[cols.index(c)], 3) for c in ACTIONABLE if c in numeric})
pd.DataFrame(rows).set_index(["contrafactual", "variable"])

### Task 3.5 · Análisis con los resultados

#### 3.5.1 · Relación entre $N_{ef}$ y Jaccard

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for sigma_rel in SIGMA_VALUES:
    ax.plot(neff_table[sigma_rel], jaccard_table[sigma_rel], marker="o", label=f"sigma_rel = {sigma_rel}")
    for n_samples in N_VALUES:
        ax.annotate(f"N={n_samples}", (neff_table.loc[n_samples, sigma_rel], jaccard_table.loc[n_samples, sigma_rel]),
                    fontsize=7, xytext=(3, -8), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("N_ef promedio (escala logarítmica)")
ax.set_ylabel("Jaccard promedio")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Para estimar cuántas muestras se necesitan para un Jaccard de 0.9 con $\sigma_{rel} = 0.75$ se ajusta una ley de potencia a la inestabilidad, $1 - J = a N^{-b}$, que es una recta en escala log-log, con los tres puntos de la tabla. Después se verifica la estimación con 20 corridas adicionales.

In [ ]:
j_075 = jaccard_table[0.75].astype(float)
instability = np.clip(1 - j_075.values, 1e-3, None)
slope, intercept = np.polyfit(np.log(N_VALUES), np.log(instability), 1)
b, a = -slope, np.exp(intercept)
print(f"Ajuste: 1 - J = {a:.4g} * N^(-{b:.4f})")
if b <= 0:
    print("La inestabilidad no disminuye con N; la extrapolación no es válida.")
else:
    n_target = (a / 0.1) ** (1 / b)
    n_verify = int(min(np.ceil(n_target / 100) * 100, 500_000))
    print(f"N estimado para Jaccard = 0.9: {n_target:,.0f} (se verifica con N = {n_verify:,})")
    j_verify, neff_verify = lime_stability(n_verify, 0.75)
    print(f"Verificación: Jaccard = {j_verify:.4f}, N_ef = {neff_verify:,.0f}")

#### 3.5.2 · Mayor discrepancia entre LIME y SHAP

Como LIME explica probabilidades y SHAP explica log-odds, se comparan las importancias relativas, $\frac{|w_{j}|}{\sum_{k}|w_{k}|}$ frente a $\frac{|\phi_{j}|}{\sum_{k}|\phi_{k}|}$, y se toma el cliente y la variable con mayor diferencia absoluta.

In [ ]:
gaps = pd.DataFrame({
    label: np.abs(lime_results[label]["coef_full"]) / np.abs(lime_results[label]["coef_full"]).sum()
    - np.abs(phi[i]) / np.abs(phi[i]).sum()
    for label, i in clients.items()
}, index=cols)
client_gap, var_disc = gaps.abs().stack().idxmax()[::-1]
i_disc, j_disc = clients[client_gap], cols.index(var_disc)
r = lime_results[client_gap]
print(f"Mayor discrepancia: cliente {client_gap}, variable {var_disc}")
print(f"  importancia relativa LIME - SHAP = {gaps.loc[var_disc, client_gap]:+.4f}")
print(f"  w_LIME = {r['coef_full'][j_disc]:+.4f} (probabilidad) | phi_SHAP = {phi[i_disc, j_disc]:+.4f} (log-odds)")
print(f"  valor del cliente = {X_test_display.iloc[i_disc, j_disc]}")

**Prueba 1.** Se repite LIME para ese cliente con varios valores de $\sigma_{rel}$ y se observa si la discrepancia cambia.

In [ ]:
phi_share = np.abs(phi[i_disc, j_disc]) / np.abs(phi[i_disc]).sum()
rows = []
for sigma_rel in (0.25, 0.5, 0.75, 1.5):
    res = lime_explain(f_prob, X_test_arr[i_disc], X_train_arr, n_samples=5000, sigma_rel=sigma_rel,
                       rng=np.random.default_rng(SEED))
    w_share = np.abs(res["coef_full"][j_disc]) / np.abs(res["coef_full"]).sum()
    rows.append({"sigma_rel": sigma_rel, "w_LIME": res["coef_full"][j_disc],
                 "importancia relativa LIME": w_share, "importancia relativa SHAP": phi_share,
                 "discrepancia": w_share - phi_share, "R²_w (K=5)": res["r2_w"], "N_ef": res["n_eff"]})
pd.DataFrame(rows).set_index("sigma_rel")

**Prueba 2.** Se grafica la probabilidad de `good` del cliente al variar solo esa variable entre los valores observados en entrenamiento (curva ICE). Como referencia se agrega el promedio de esa curva sobre el fondo de 100 instancias (dependencia parcial). Si la curva del cliente difiere en forma de la curva promedio, hay indicios de interacción.

In [ ]:
values = np.unique(X_train_arr[:, j_disc])
if values.size > 60:
    values = np.unique(np.quantile(X_train_arr[:, j_disc], np.linspace(0, 1, 60)))

grid = np.repeat(X_test_arr[i_disc][None], values.size, axis=0)
grid[:, j_disc] = values
ice = f_prob(grid)

bg_arr = background.values
pdp = np.array([f_prob(np.where(np.arange(len(cols)) == j_disc, v, bg_arr)).mean() for v in values])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(values, ice, marker="o", ms=3, label=f"Cliente {client_gap}")
ax.plot(values, pdp, ls="--", color="gray", label="Promedio sobre el fondo")
ax.axvline(X_test_arr[i_disc, j_disc], color="red", lw=1, label="Valor del cliente")
ax.axhline(THETA, color="black", lw=0.5)
ax.set_xlabel(f"{var_disc} (código)" if var_disc in categorical else var_disc)
ax.set_ylabel("Probabilidad de good")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
if var_disc in categorical:
    print("Códigos:", dict(enumerate(encoder.categories_[categorical.index(var_disc)])))
print(f"Rango de la curva del cliente: {ice.min():.4f} a {ice.max():.4f} | rango del promedio: {pdp.min():.4f} a {pdp.max():.4f}")

#### 3.5.4 · Contrafactuales frente a SHAP en el cliente rechazado

Para las tres variables de mayor $|\phi|$ del cliente rechazado se indica si son accionables y se calcula el mejor cambio individual posible, probando todos los valores observados en entrenamiento y cambiando solo esa variable.

In [ ]:
changed_vars = sorted({cols[j] for idx in chosen for j in np.flatnonzero(cands[idx] != x_bad)})
top3_bad = [cols[j] for j in np.argsort(-np.abs(phi[i_bad]))[:3]]
print(f"Variables que cambian en los contrafactuales: {changed_vars}")
print(f"Tres variables de mayor |phi| del cliente rechazado: {top3_bad}")

rows = []
for var in top3_bad:
    j = cols.index(var)
    options = np.unique(X_train_arr[:, j])
    grid = np.repeat(x_bad[None], options.size, axis=0)
    grid[:, j] = options
    probs = f_prob(grid)
    dists = distance(x_bad, grid, numeric_idx, categorical_idx, scale)
    best = int(np.argmax(probs))
    rows.append({
        "variable": var,
        "phi_j": phi[i_bad, j],
        "tipo": "inmutable" if var in IMMUTABLE else ("accionable" if var in ACTIONABLE else "no accionable"),
        "valor actual": decode(x_bad[j], var, encoder, categorical),
        "mejor valor": decode(options[best], var, encoder, categorical),
        "p(good) máxima": probs[best],
        "distancia del mejor valor": dists[best],
        "MAD": scale[j] if var in numeric else np.nan,
    })
pd.DataFrame(rows).set_index("variable")